# Beginner Data Pipeline — Step-by-Step Explanation

This note explains the **end-to-end pipeline** implemented in `beginner_data_pipeline.py`. It follows a classic **ETL/ELT** pattern and keeps dependencies light (CSV + SQLite only).

---

## 0) Overview & Outputs

**Goal:** simulate market OHLCV data → validate → clean/align → engineer features → save curated datasets → enable simple queries and a quick plot.

**Default output folders**
- Raw (bronze): `/mnt/data/pipeline_demo_v2/raw/*.csv`
- Silver (clean): `/mnt/data/pipeline_demo_v2/silver/combined.csv`
- Gold (features): `/mnt/data/pipeline_demo_v2/gold/features.csv`
- Mini feature store (SQLite): `/mnt/data/pipeline_demo_v2/feature_store.sqlite`
- Summary report: `/mnt/data/pipeline_demo_v2/report_summary.txt`

**Run**
```bash
python beginner_data_pipeline.py
# or choose another output directory:
python beginner_data_pipeline.py --outdir ./my_outputs



## 1) Extract (make raw data)
- Simulate ~250 business days of OHLCV for 3 tickers (`AAA, BBB, CCC`) with a simple random-walk.
- Save **one CSV per ticker** to the **raw/ (bronze)** folder.

**Output:** `raw/AAA_ohlcv.csv`, `raw/BBB_ohlcv.csv`, `raw/CCC_ohlcv.csv`

---

## 2) Validate (basic data quality)
- Check for:
  - missing values  
  - duplicate timestamps  
  - timestamps increasing within each ticker  
  - positive prices and consistent OHLC (high ≥ max(open, close); low ≤ min(open, close))

**Goal:** stop early if raw data is broken.

---

## 3) Transform → Silver (clean table)
- Concatenate the 3 raw CSVs.
- Sort by (`ticker`, `timestamp`) and drop duplicates.
- Forward/backward fill within each ticker to remove tiny gaps.
- Save a **single clean table** to **silver/**.

**Output:** `silver/combined.csv`

---

## 4) Feature Engineering → Gold (model-ready table)
Compute simple, useful features per ticker:
- Daily returns: `ret_1d`, 5-day returns: `ret_5d`
- Momentum (20-day): `mom_20 = close/close.shift(20) - 1`
- Annualized volatility (20-day): `vol_20 = std(rolling 20 of ret_1d) * sqrt(252)`
- Moving averages: `sma_20`, `sma_50`, and `sma_ratio = sma_20 / sma_50`
- Label (for ML): `label_ret_tplus1` = next-day return

Save the curated panel to **gold/**.

**Output:** `gold/features.csv`

---

## 5) Load (mini “feature store”)
- Write the gold table into a **SQLite** DB with an index on (`ticker`, `timestamp`) for fast queries.

**Output:** `feature_store.sqlite` (table: `features`)

---

## 6) Serve (tiny helpers)
- `get_features_for(ticker, start, end)` → returns features for a date window.  
- `latest_feature_vector(ticker)` → returns the most recent row for that ticker.

**Use case:** quick retrieval for backtests, notebooks, or a simple API later.

---

## 7) Report & Quick Visual
- Write a small **text summary** (rows, date range, mean daily return, avg 20-day annualized vol) per ticker.
- Plot **Close vs SMA20/SMA50** for the last ~60 days (sanity check).

**Outputs:** `report_summary.txt` + one preview chart

---

## Run it
```bash
python beginner_data_pipeline.py
# or choose another output folder:
python beginner_data_pipeline.py --outdir ./my_outputs


In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
# beginner_data_pipeline.py
# ------------------------------------------------------------
# End-to-end, dependency-light data pipeline 
# - EXTRACT: Generate synthetic OHLCV for 3 tickers -> raw CSVs
# - VALIDATE: Basic data-quality checks
# - TRANSFORM: Clean + compute features (returns, SMA, volatility)
# - LOAD: Save curated CSVs ("silver", "gold") + SQLite "feature store"
# - SERVE: Helper functions to query features by ticker & date
# - REPORT: Summary text + small preview plot
#
# Outputs (default):
#   /mnt/data/pipeline_demo_v2/raw/*.csv
#   /mnt/data/pipeline_demo_v2/silver/combined.csv
#   /mnt/data/pipeline_demo_v2/gold/features.csv
#   /mnt/data/pipeline_demo_v2/feature_store.sqlite
#   /mnt/data/pipeline_demo_v2/report_summary.txt
# ------------------------------------------------------------

import os
import argparse
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

# ---------- CONFIG ----------
np.random.seed(7)
OUTDIR = os.environ.get("PIPELINE_OUTDIR", "/mnt/data/pipeline_demo_v2")
RAW_DIR = os.path.join(OUTDIR, "raw")       # bronze
SILVER_DIR = os.path.join(OUTDIR, "silver") # silver
GOLD_DIR = os.path.join(OUTDIR, "gold")     # gold
DB_PATH = os.path.join(OUTDIR, "feature_store.sqlite")
TICKERS = ["AAA", "BBB", "CCC"]
N_DAYS = 250  # ~1Y of business days

for d in [OUTDIR, RAW_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)

def log(msg: str):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")

def bdate_range(end_date=None, periods=252):
    if end_date is None:
        end_date = pd.Timestamp.today(tz="UTC").normalize()
    rng = pd.bdate_range(end=end_date, periods=periods)
    return rng.tz_localize(None)

# ---------- 1) EXTRACT ----------
def simulate_ohlcv(ticker: str, dates: pd.DatetimeIndex) -> pd.DataFrame:
    """Geometric-Brownian-like synthetic OHLCV."""
    mu = 0.10      # annual drift
    sigma = 0.25   # annual vol
    dt = 1/252

    z = np.random.normal(size=len(dates))
    log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*z
    price = 100*np.exp(np.cumsum(log_ret))

    close = pd.Series(price, index=dates, name="close")
    high = close * (1 + np.random.uniform(0.001, 0.02, size=len(dates)))
    low  = close * (1 - np.random.uniform(0.001, 0.02, size=len(dates)))
    open_ = close.shift(1).fillna(close.iloc[0])
    vol = (np.abs(np.random.normal(loc=1e6, scale=1e5, size=len(dates)))).astype(int)

    df = pd.DataFrame({
        "timestamp": dates,
        "ticker": ticker,
        "open": open_.values,
        "high": high.values,
        "low":  low.values,
        "close": close.values,
        "volume": vol
    })
    # Ensure OHLC consistency
    df["high"] = df[["open","close","high"]].max(axis=1)
    df["low"]  = df[["open","close","low"]].min(axis=1)
    return df.reset_index(drop=True)

def extract_raw() -> list:
    log("Extract: generating synthetic OHLCV → raw CSVs")
    dates = bdate_range(periods=N_DAYS)
    paths = []
    for t in TICKERS:
        df = simulate_ohlcv(t, dates)
        path = os.path.join(RAW_DIR, f"{t}_ohlcv.csv")
        df.to_csv(path, index=False)
        paths.append(path)
        log(f"  wrote {path} ({len(df)} rows)")
    return paths

# ---------- 2) VALIDATE ----------
@dataclass
class ValidationResult:
    passed: bool
    issues: list

def validate_raw(paths) -> ValidationResult:
    log("Validate: basic checks on raw CSVs")
    issues = []
    for path in paths:
        df = pd.read_csv(path, parse_dates=["timestamp"])
        # Basic checks
        if df.isna().sum().sum() > 0:
            issues.append((path, "Null values present"))
        if df.duplicated(subset=["timestamp"]).any():
            issues.append((path, "Duplicate timestamps"))
        if not df["timestamp"].is_monotonic_increasing:
            issues.append((path, "Timestamps not monotonic increasing"))
        if (df[["open","high","low","close"]] <= 0).any().any():
            issues.append((path, "Non-positive prices"))
        if (df["high"] < df[["open","close"]].max(axis=1)).any():
            issues.append((path, "High < max(open,close)"))
        if (df["low"]  > df[["open","close"]].min(axis=1)).any():
            issues.append((path, "Low > min(open,close)"))
    passed = len(issues) == 0
    if passed:
        log("  ✓ All checks passed")
    else:
        for p, msg in issues:
            log(f"  ✗ {os.path.basename(p)}: {msg}")
    return ValidationResult(passed=passed, issues=issues)

# ---------- 3) TRANSFORM ----------
def to_silver(paths) -> pd.DataFrame:
    log("Transform (→ silver): concat & clean")
    frames = [pd.read_csv(p, parse_dates=["timestamp"]) for p in paths]
    all_df = pd.concat(frames, ignore_index=True).sort_values(["ticker","timestamp"])
    all_df = all_df.drop_duplicates(subset=["ticker","timestamp"])
    # Forward/backward fill within each ticker; suppress future groupby warning
    all_df[["open","high","low","close","volume"]] = (
        all_df.groupby("ticker", group_keys=False)[["open","high","low","close","volume"]]
             .apply(lambda g: g.ffill().bfill())
    )
    silver_path_csv = os.path.join(SILVER_DIR, "combined.csv")
    all_df.to_csv(silver_path_csv, index=False)
    log(f"  wrote {silver_path_csv} ({len(all_df)} rows)")
    return all_df

def compute_features(silver_df: pd.DataFrame) -> pd.DataFrame:
    log("Transform (→ gold): compute features (returns, momentum, vol, SMAs)")
    df = silver_df.copy()
    df["ret_1d"] = df.groupby("ticker")["close"].pct_change()
    df["ret_5d"] = df.groupby("ticker")["close"].pct_change(5)
    df["mom_20"] = df.groupby("ticker")["close"].transform(lambda s: s/s.shift(20) - 1)
    df["vol_20"] = df.groupby("ticker")["ret_1d"].transform(lambda s: s.rolling(20).std() * np.sqrt(252))
    df["sma_20"] = df.groupby("ticker")["close"].transform(lambda s: s.rolling(20).mean())
    df["sma_50"] = df.groupby("ticker")["close"].transform(lambda s: s.rolling(50).mean())
    df["sma_ratio"] = df["sma_20"] / df["sma_50"]
    # Simple label for ML: next-day return
    df["label_ret_tplus1"] = df.groupby("ticker")["ret_1d"].shift(-1)

    gold_cols = [
        "timestamp","ticker","close","volume",
        "ret_1d","ret_5d","mom_20","vol_20","sma_20","sma_50","sma_ratio","label_ret_tplus1"
    ]
    gold = df[gold_cols].dropna().reset_index(drop=True)

    gold_path_csv = os.path.join(GOLD_DIR, "features.csv")
    gold.to_csv(gold_path_csv, index=False)
    log(f"  wrote {gold_path_csv} ({len(gold)} rows)")
    return gold

# ---------- 4) LOAD → SQLite mini "feature store" ----------
def load_to_sqlite(gold_df: pd.DataFrame, db_path: str = DB_PATH):
    log("Load: writing features to SQLite")
    conn = sqlite3.connect(db_path)
    gold_df.to_sql("features", conn, if_exists="replace", index=False)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_features_ticker_time ON features(ticker, timestamp);")
    conn.commit()
    conn.close()
    log(f"  wrote table 'features' to {db_path}")

# ---------- 5) SERVE (helper queries) ----------
def get_features_for(ticker: str, start: str, end: str, db_path: str = DB_PATH) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    q = """
    SELECT * FROM features
    WHERE ticker = ?
      AND timestamp BETWEEN ? AND ?
    ORDER BY timestamp
    """
    df = pd.read_sql_query(q, conn, params=(ticker, start, end), parse_dates=["timestamp"])
    conn.close()
    return df

def latest_feature_vector(ticker: str, db_path: str = DB_PATH) -> pd.Series:
    conn = sqlite3.connect(db_path)
    q = """
    SELECT * FROM features
    WHERE ticker = ?
    ORDER BY timestamp DESC
    LIMIT 1
    """
    df = pd.read_sql_query(q, conn, params=(ticker,), parse_dates=["timestamp"])
    conn.close()
    if df.empty:
        return pd.Series(dtype=float)
    return df.iloc[0]

# ---------- 6) REPORT ----------
def write_summary(gold_df: pd.DataFrame):
    kpi = gold_df.groupby("ticker").agg(
        rows=("ticker","size"),
        start=("timestamp","min"),
        end=("timestamp","max"),
        mean_ret_1d=("ret_1d","mean"),
        vol_annual=("vol_20","mean")
    ).reset_index()
    report_path = os.path.join(OUTDIR, "report_summary.txt")
    with open(report_path, "w") as f:
        f.write("Beginner Data Pipeline — Quick Summary\n")
        f.write("="*44 + "\n\n")
        f.write(kpi.to_string(index=False))
        f.write("\n")
    log(f"Summary written to {report_path}")

def preview_plot(ticker: str = "AAA", days: int = 60):
    # Use the CSV (gold) for simplicity
    gold_path_csv = os.path.join(GOLD_DIR, "features.csv")
    if not os.path.exists(gold_path_csv):
        log("No gold CSV to plot. Skipping preview plot.")
        return
    df = pd.read_csv(gold_path_csv, parse_dates=["timestamp"])
    sub = df[df["ticker"] == ticker].sort_values("timestamp").tail(days)
    if sub.empty:
        log(f"No rows to plot for {ticker}.")
        return
    sub = sub.set_index("timestamp")
    sub[["close","sma_20","sma_50"]].plot(figsize=(8,4))
    plt.title(f"{ticker} — Close vs SMA20/SMA50 (preview)")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()

# ---------- 7) MAIN ----------
def run_pipeline():
    log("=== PIPELINE START ===")
    raw_paths = extract_raw()
    val = validate_raw(raw_paths)
    if not val.passed:
        log("Validation failed. Fix issues before proceeding.")
        return
    silver = to_silver(raw_paths)
    gold = compute_features(silver)
    load_to_sqlite(gold)
    write_summary(gold)

    # Example "serve" query window: last ~60 calendar days in gold
    sample_end = gold["timestamp"].max().strftime("%Y-%m-%d")
    sample_start = (gold["timestamp"].max() - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
    fetched = get_features_for("AAA", sample_start, sample_end)
    log(f"Toy serve: fetched {len(fetched)} rows for AAA between {sample_start} and {sample_end}")

    preview_plot("AAA", 60)
    log("=== PIPELINE END ===")
    print("\nOutputs ready:")
    print(f"- Raw CSVs:           {RAW_DIR}")
    print(f"- Silver CSV:         {os.path.join(SILVER_DIR, 'combined.csv')}")
    print(f"- Gold CSV:           {os.path.join(GOLD_DIR, 'features.csv')}")
    print(f"- SQLite DB:          {DB_PATH}")
    print(f"- Summary report:     {os.path.join(OUTDIR, 'report_summary.txt')}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Beginner Data Pipeline (no external deps).")
    parser.add_argument("--outdir", type=str, default=OUTDIR, help="Output directory (default: /mnt/data/pipeline_demo_v2)")
    args = parser.parse_args()

    # Allow override via CLI flag
    if args.outdir != OUTDIR:
        global OUTDIR, RAW_DIR, SILVER_DIR, GOLD_DIR, DB_PATH
        OUTDIR = args.outdir
        RAW_DIR = os.path.join(OUTDIR, "raw")
        SILVER_DIR
